In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

os.makedirs("models", exist_ok=True)
print("Environment Ready")

Environment Ready


In [14]:
def train_and_tune_advanced(X, y, model_name):
    print(f"\nOptimizing {model_name.upper()} Model...")
    
    pos_weight = (y == 0).sum() / (y == 1).sum()
    print(f"Class Imbalance Ratio: {pos_weight:.2f}")

    param_dist = {
        'max_depth': [3, 4, 5, 6, 8],        
        'learning_rate': [0.01, 0.02, 0.05, 0.1], 
        'n_estimators': [100, 200, 300, 500],
        'gamma': [0, 0.1, 0.2, 0.5],           
        'min_child_weight': [1, 3, 5, 7],       
        'subsample': [0.6, 0.8, 1.0],   
        'colsample_bytree': [0.6, 0.8, 1.0], 
        'scale_pos_weight': [pos_weight, pos_weight * 1.5],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [1, 1.5, 2, 5]
    }

    xgb_clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist', 
        n_jobs=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    search = RandomizedSearchCV(
        estimator=xgb_clf,
        param_distributions=param_dist,
        n_iter=100,
        scoring='roc_auc',
        cv=cv,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    search.fit(X, y)

    # 5. Report
    best_model = search.best_estimator_
    print(f"Best ROC-AUC: {search.best_score_:.4f}")
    
    # 6. Save
    best_model.get_booster().feature_names = list(X.columns)
    best_model.save_model(f"models/{model_name}_optimized.json")
    print(f"Saved to models/{model_name}_optimized.json")
    
    return best_model

In [15]:
print("Fetching Kidney Data...")
ckd = fetch_ucirepo(id=336)
X_ckd = ckd.data.features.copy()
y_ckd = ckd.data.targets['class'].copy()

rename_map = {
    'age': 'AGE', 'bp': 'LAB_BP', 'sg': 'LAB_URINE_SG', 'al': 'LAB_URINE_ALBUMIN',
    'su': 'LAB_URINE_SUGAR', 'rbc': 'LAB_URINE_RBC', 'pc': 'LAB_URINE_PUS',
    'pcc': 'LAB_URINE_CLUMPS', 'ba': 'LAB_URINE_BACTERIA', 'bgr': 'LAB_GLUCOSE_RANDOM',
    'bu': 'LAB_BUN', 'sc': 'LAB_CREATININE', 'sod': 'LAB_SODIUM', 'pot': 'LAB_POTASSIUM',
    'hemo': 'LAB_HEMOGLOBIN', 'pcv': 'LAB_HCT', 
    'wc': 'LAB_WBC', 'wbcc': 'LAB_WBC', 
    'rc': 'LAB_RBC', 'rbcc': 'LAB_RBC',
    'htn': 'HISTORY_HTN', 'dm': 'HISTORY_DM', 'cad': 'HISTORY_CAD',
    'appet': 'SYMPTOM_APPETITE', 'pe': 'SYMPTOM_EDEMA', 'ane': 'SYMPTOM_ANEMIA'
}
X_ckd = X_ckd.rename(columns=rename_map)

text_map = {
    'normal': 0, 'abnormal': 1, 'notpresent': 0, 'present': 1,
    'no': 0, 'yes': 1, 'good': 0, 'poor': 1, '\tno': 0, '\tyes': 1,
    ' ckd': 1, 'ckd': 1, 'notckd': 0
}
for col in X_ckd.columns:
    if X_ckd[col].dtype == 'object':
        X_ckd[col] = X_ckd[col].astype(str).str.lower().str.strip().map(text_map)
    X_ckd[col] = pd.to_numeric(X_ckd[col], errors='coerce')

y_ckd = y_ckd.astype(str).str.lower().str.strip().map(text_map).fillna(0).astype(int)

best_kidney = train_and_tune_advanced(X_ckd, y_ckd, "kidney")

Fetching Kidney Data...

Optimizing KIDNEY Model...
Class Imbalance Ratio: 0.60
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best ROC-AUC: 1.0000
Saved to models/kidney_optimized.json


In [16]:
print("\nFetching Liver Data...")
ilpd = fetch_ucirepo(id=225)
X_liver = ilpd.data.features.copy()
y_liver = ilpd.data.targets['Selector'].copy()

X_liver = X_liver.rename(columns={
    'Age': 'AGE', 'Gender': 'GENDER', 
    'TB': 'LAB_BILIRUBIN_TOTAL', 'DB': 'LAB_BILIRUBIN_DIRECT', 
    'Alkphos': 'LAB_ALP', 'Sgpt': 'LAB_SGPT', 'Sgot': 'LAB_SGOT', 
    'TP': 'LAB_PROTEIN', 'ALB': 'LAB_ALBUMIN', 'A/G Ratio': 'LAB_AG_RATIO'
})

X_liver['GENDER'] = X_liver['GENDER'].apply(lambda x: 0 if x == 'Female' else 1)
y_liver = y_liver.apply(lambda x: 1 if x == 1 else 0)

X_liver['LAB_BILIRUBIN_INDIRECT'] = X_liver['LAB_BILIRUBIN_TOTAL'] - X_liver['LAB_BILIRUBIN_DIRECT']

X_liver['LAB_GLOBULIN'] = X_liver['LAB_PROTEIN'] - X_liver['LAB_ALBUMIN']

X_liver['LAB_AG_RATIO'] = X_liver['LAB_AG_RATIO'].fillna(
    X_liver['LAB_ALBUMIN'] / (X_liver['LAB_GLOBULIN'] + 1e-5)
)

X_liver['LAB_SGOT_SGPT_RATIO'] = X_liver['LAB_SGOT'] / (X_liver['LAB_SGPT'] + 1e-5)

for col in ['LAB_SGPT', 'LAB_SGOT', 'LAB_ALP', 'LAB_BILIRUBIN_TOTAL']:
    upper_limit = X_liver[col].quantile(0.99)
    X_liver[col] = X_liver[col].clip(upper=upper_limit)

best_liver = train_and_tune_advanced(X_liver, y_liver, "liver")


Fetching Liver Data...


/tmp/ipykernel_373154/2832404101.py:28: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_liver[col] = X_liver[col].clip(upper=upper_limit)



Optimizing LIVER Model...
Class Imbalance Ratio: 0.40
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best ROC-AUC: 0.7542
Saved to models/liver_optimized.json


In [17]:
print("\nVerifying...")
for name in ["kidney", "liver"]:
    path = f"models/{name}_optimized.json"
    if os.path.exists(path):
        bst = xgb.Booster()
        bst.load_model(path)
        print(f"{name.upper()} Model Loaded. Feature Count: {len(bst.feature_names)}")
    else:
        print(f"{name.upper()} Model NOT Found.")


Verifying...
KIDNEY Model Loaded. Feature Count: 24
LIVER Model Loaded. Feature Count: 13
